# 02_prepare_towers Prepare OpenCelliD towers

**Owner:** Person 1

**Goal:** Filter Vietnam cells to 4G/5G cells inside Hanoi, merge them into towers and check coverage.
4G and 5G are not separated: 5G mostly shares sites with 4G and OpenCelliD has very few NR cells.

**Outputs:** data/interim/hanoi_towers.csv

In [ ]:
import sys
from pathlib import Path

# Cho phép import package speedmap khi chạy notebook từ thư mục notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from speedmap import config

## 1. Load the raw OpenCelliD file
Download `452.csv.gz` from opencellid.org into `data/raw/` first.

In [ ]:
from speedmap.data import towers

raw_towers = towers.load_opencellid()
print(len(raw_towers), "cells in Vietnam")
raw_towers.radio.value_counts()

## 2. Filter to Hanoi

In [ ]:
hanoi = towers.filter_hanoi(raw_towers)
print(len(hanoi), "towers in Hanoi from", hanoi.cells.sum(), "cells")
hanoi.head()

## 3. Check coverage
Plot the towers over the Ookla tiles and measure how far each tile is from the nearest tower.
Towers are dense in the city centre and sparse in the suburbs; we keep all tiles, because the network outside the centre is also slower in reality.

In [ ]:
from speedmap.data import ookla

tiles = ookla.load_raw().drop_duplicates("quadkey")

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(tiles.longitude, tiles.latitude, s=2, c="lightgrey", label="Ookla tiles")
ax.scatter(hanoi.longitude, hanoi.latitude, s=4, c="#1c7ed6", label="Towers (4G/5G)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.set_title("OpenCelliD towers over Ookla tiles, Hanoi")
ax.legend(); plt.show()

# Khoảng cách từ mỗi ô tới trạm gần nhất (haversine, đủ nhanh với ~10k ô x ~2k trạm)
lat1, lon1 = np.radians(tiles.latitude.to_numpy())[:, None], np.radians(tiles.longitude.to_numpy())[:, None]
lat2, lon2 = np.radians(hanoi.latitude.to_numpy())[None, :], np.radians(hanoi.longitude.to_numpy())[None, :]
dist = np.empty(len(tiles))
for i in range(0, len(tiles), 500):
    a = (np.sin((lat2 - lat1[i:i + 500]) / 2) ** 2
         + np.cos(lat1[i:i + 500]) * np.cos(lat2) * np.sin((lon2 - lon1[i:i + 500]) / 2) ** 2)
    dist[i:i + 500] = (2 * 6_371_000 * np.arcsin(np.sqrt(a))).min(axis=1)

for r in [500, 1000, 2000, 5000]:
    print(f"tiles with a tower within {r} m: {(dist <= r).mean():.1%}")
print(f"median distance to nearest tower: {np.median(dist):.0f} m")

## 4. Save

In [ ]:
config.TOWERS_FILE.parent.mkdir(parents=True, exist_ok=True)
hanoi.to_csv(config.TOWERS_FILE, index=False)
print("saved", config.TOWERS_FILE)